# Fed-BioMed Researcher to train a model on a CSV dataset

This example shows how to use a CSV format file as a node dataset. The example CSV file is synthetic data with a format inspired from ADNI dataset.

This example uses Pseudo Adni Dataset. Please check `README.md` file in `notebooks` directory for the instructions to load Pseudo Adni dataset and configure nodes.

## Create an experiment to train a model on the data found

Declare a torch training plan MyTrainingPlan class to send for training on the node

In [ ]:
import torch
import torch.nn as nn
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import TabularDataset
import pandas as pd

# Here we define the model to be used. 
# You can use any class name (here 'MyTrainingPlan')
class MyTrainingPlan(TorchTrainingPlan):
    
    # Model 
    def init_model(self, model_args):    
        model = self.Net(model_args)
        return model 
    
    
    # Dependencies
    def init_dependencies(self):
        # Here we define the custom dependencies that will be needed by our custom Dataloader
        # In this case, we need the torch Dataset and DataLoader classes
        # We need pandas to read the local .csv file at the node side
        deps = ["from fedbiomed.common.dataset import TabularDataset"]
        
        return deps
    
    class Net(nn.Module):
        def __init__(self, model_args):
            super().__init__()
            # should match the model arguments dict passed below to the experiment class
            self.fc1 = nn.Linear(model_args['in_features'], 5)
            self.fc2 = nn.Linear(5, model_args['out_features'])

        def forward(self, x):
            x = self.fc1(x)
            x = F.relu(x)
            x = self.fc2(x)
            return x

    def training_step(self, data, target):
        output = self.model().forward(data).float()
        criterion = torch.nn.MSELoss()
        loss   = criterion(output, target.unsqueeze(1))
        return loss

    def training_data(self):
        x_dim = self.model_args()['in_features']
        dataset = TabularDataset(
            input_columns=list(range(1,x_dim+1)),
            target_columns='APOE4',
            transform=lambda xs: torch.as_tensor(xs, dtype=torch.float32),
            target_transform=lambda xs: torch.as_tensor(xs, dtype=torch.float32)
        )

        train_kwargs = {'shuffle': True}
        
        return DataManager(dataset=dataset, **train_kwargs)

In [ ]:
# model parameters 
model_args = {
    'in_features': 15, 
    'out_features': 1
}

# training parameters 
training_args = {
    'loader_args': { 'batch_size': 20, }, 
    'optimizer_args': {
        'lr': 1e-3
    }, 
    'epochs': 2, 
    'dry_run': False,  
}

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

# Calling the training data with specified tags
tags =  ['adni']
rounds = 2

exp = Experiment(tags=tags,
                 training_plan_class=MyTrainingPlan,
                 model_args=model_args,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 node_selection_strategy=None,
                 save_breakpoints=True,
                 tensorboard=True
                )

In [ ]:
tensorboard_dir = exp.tensorboard_results_path
tensorboard_dir

In [ ]:
# Uncomment for tensorboard 
#%load_ext tensorboard
#%tensorboard --logdir "$tensorboard_dir"

In [ ]:
#exp.training_plan_file()

### Set pre-processing

In [ ]:
from fedbiomed.common.constants import PreprocType

# Caveat: pre-processing uses the whole dataset, even if training plan is filtering
# only some columns for training. So column numbers reference the *full* dataset
fedcombat_args = {
    # Can use numeric values
    #'covariates': [1,2],
    #'phenotypes': [3,4,5,6,7,8,9,10,11,12,13,14,15],

    # Can use label values
    'covariates': ['SEX','AGE','PTEDUCAT'],
    'phenotypes': [
        'CDRSB.bl','ADAS11.bl','MMSE.bl','RAVLT.immediate.bl','RAVLT.learning.bl','RAVLT.forgetting.bl',
        'FAQ.bl','WholeBrain.bl','Ventricles.bl','Hippocampus.bl','MidTemp.bl','Entorhinal.bl'
    ],

    # Can NOT mix numeric and label in same variable
    #'covariates': [1,'AGE','PTEDUCAT'],
    #'phenotypes': ['CDRSB.bl','ADAS11.bl',6,7],

    # Could mix numeric and label in different variable
    #'covariates': ['SEX','AGE','PTEDUCAT'],
    #'phenotypes': [4,5,6,7],
}

# associate a pre-processing with the experiment
# by default, when creating experiment, no pre-processing is configured
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)

In [ ]:
# Optionally test node filter
#exp.set_nodes(['replace_with_some_experiment_NODE_ID'])
#exp.run_once(increase=True)

In [ ]:
# Optionally remove node filter 
#exp.set_nodes(None)
#exp.run_once(increase=True)

In [ ]:
# first round: needs pre-preprocessing
exp.run_once(increase=True)

In [ ]:
# same context: does not need pre-processing again ...
exp.run_once(increase=True)

In [ ]:
# ... yet preprocessing is associated with the experiment
# exp.preprocessing().__dict_

In [ ]:
# set again preprocessing: needs pre-processing - event with same conditions
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)

In [ ]:
exp.run_once(increase=True)

In [ ]:
import copy
fedcombat_args2 = copy.deepcopy(fedcombat_args)
fedcombat_args2.update(
    {
        'rounds': 5,  # non-default number of training rounds for the harmonization model
        'training_args': {
            'loader_args': {'batch_size': 8},
            'log_interval': 10,  # reasonably verbose harmonization logging
        }
    }
)

In [ ]:
# set again preprocessing: needs pre-processing
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args2)

In [ ]:
# pre-processing can also be triggered without training ...
exp.preprocessing().execute()

In [ ]:
# ... still it is run only if needed
exp.preprocessing().execute()

In [ ]:
exp.run_once(increase=True)

In [ ]:
# no pre-processing
exp.set_preprocessing(PreprocType.NONE)
exp.run_once(increase=True)

In [ ]:
exp.preprocessing()

In [ ]:
# set again preprocessing: needs pre-processing
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)
exp.run_once(increase=True)

In [ ]:
# check full pre-processing config at time of saving breakpoint
#exp.preprocessing().__dict__

### Load breakpoint

In [ ]:
del exp

In [ ]:
loaded_exp = Experiment.load_breakpoint()

In [ ]:
# check pre-processing config after loading breakpoint : same configuration as the one saved
#loaded_exp.preprocessing().__dict__

In [ ]:
# Tensorboard needs to be manually set again after loading a breakpoint
#loaded_exp.set_tensorboard(True)

In [ ]:
loaded_exp.run_once(increase=True)

### Save trained model to file

In [ ]:
exp = loaded_exp

In [ ]:
try: 
    exp.training_plan().export_model('./trained_model')
except Exception as e:
    print(e)

## Default notebook experiment

Let's start the experiment.

By default, this function doesn't stop until all the `round_limit` rounds are done for all the nodes

In [ ]:
#exp.run()

Save trained model to file

In [ ]:
exp.training_plan().export_model('./trained_model')

In [ ]:
print("\nList the training rounds : ", exp.training_replies().keys())

print("\nList the nodes for the last training round and their timings : ")
round_data = exp.training_replies()[rounds - 1]
for r in round_data.values():
    print("\t- {id} :\
    \n\t\trtime_training={rtraining:.2f} seconds\
    \n\t\tptime_training={ptraining:.2f} seconds\
    \n\t\trtime_total={rtotal:.2f} seconds".format(id = r['node_id'],
        rtraining = r['timing']['rtime_training'],
        ptraining = r['timing']['ptime_training'],
        rtotal = r['timing']['rtime_total']))
print('\n')


In [ ]:
print("\nList the training rounds : ", exp.aggregated_params().keys())
print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())
